# Shape Classification with MediculaMultiTaskModel (MeViS)

This notebook loads the `MediculaMultiTaskModel` from MeViS, runs inference on a directory of shape images (triangles, squares, circles, etc.), and compares predictions against a `labels.csv` ground-truth file.

**Expected CSV format (`labels.csv`):**
```
image_name,label
circle_001.png,circle
square_042.png,square
triangle_007.png,triangle
```

## 1. Install Dependencies

In [4]:
# Run this cell once to install everything needed
import subprocess, sys

packages = [
    "medicalmultitaskmodeling[interactive]",  # MMM — NativeBlocks
    "numpy",
    "pandas",
    "matplotlib",
    "Pillow",
    "scikit-learn",
]

for pkg in packages:
    print(f"Installing {pkg}...", end=" ")
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", pkg, "-q"],
        capture_output=True, text=True
    )
    print("\u2705" if result.returncode == 0 else f"\u274c {result.stderr.strip()}")

# cv2 is a system dep — if import fails run the line below
print("\nIf cv2 is missing: !sudo apt install python3-opencv -y")
print("All done!")


Installing medicalmultitaskmodeling[interactive]... ❌ ERROR: Ignored the following versions that require a different python version: 1.0.3 Requires-Python >=3.10,<3.12; 1.0.4 Requires-Python >=3.10,<3.12
ERROR: Could not find a version that satisfies the requirement medicalmultitaskmodeling[interactive] (from versions: none)
ERROR: No matching distribution found for medicalmultitaskmodeling[interactive]

Installing numpy... ✅
Installing pandas... ✅
Installing matplotlib... ✅
Installing Pillow... ✅
Installing scikit-learn... ✅

If cv2 is missing: !sudo apt install python3-opencv -y
All done!


## 2. Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms

# MMM imports
try:
    from mmm.labelstudio_ext.NativeBlocks import NativeBlocks, MMM_MODELS, DEFAULT_MODEL
    print("\u2705 MMM imported successfully")
    print(f"Available models: {list(MMM_MODELS.keys())}")
    print(f"Default model: {DEFAULT_MODEL}")
except ImportError as e:
    print(f"\u274c MMM import failed: {e}")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")


## 3. Configuration

In [ ]:
# NativeBlocks wraps the MMM encoder + task heads.
# The model downloads automatically on first run (~few GB).
model = NativeBlocks(MMM_MODELS[DEFAULT_MODEL], device_identifier=f"{DEVICE}")

encoder  = model["encoder"]
squeezer = model["squeezer"]  # pools the feature pyramid -> single hidden vector

# Quick sanity check — forward a dummy image
with torch.inference_mode():
    dummy = torch.rand(1, 3, 224, 224).to(model.device)
    feature_pyramid = encoder(dummy)           # list of Tensors
    hidden = nn.Flatten(1)(squeezer(feature_pyramid)[1])  # (1, D)

print(f"\u2705 Model loaded | hidden dim: {hidden.shape[-1]}")

# --- Classification head --------------------------------------------------
# MMM encoder produces feature embeddings; we add a linear head for shapes.
# If you have a fine-tuned checkpoint, load it below instead.
EMBED_DIM = hidden.shape[-1]
clf_head = nn.Linear(EMBED_DIM, NUM_CLASSES).to(model.device)
# clf_head.load_state_dict(torch.load('clf_head.pth'))  # uncomment if you have weights
clf_head.eval()
print(f"Classification head: {EMBED_DIM} -> {NUM_CLASSES}")


## 4. Load Labels CSV

In [ ]:
labels_df = pd.read_csv(labels_csv)

# Normalise column names (strip whitespace, lowercase)
labels_df.columns = [c.strip().lower() for c in labels_df.columns]

# Accept 'image_name' or 'filename' or 'image'
name_col = next((c for c in labels_df.columns if 'image' in c or 'file' in c or 'name' in c), labels_df.columns[0])
label_col = next((c for c in labels_df.columns if 'label' in c or 'class' in c), labels_df.columns[1])

labels_df = labels_df.rename(columns={name_col: 'image_name', label_col: 'label'})
label_map = dict(zip(labels_df['image_name'], labels_df['label']))

print(f"Loaded {len(label_map)} label entries")
print(f"Unique classes in CSV: {sorted(labels_df['label'].unique())}")
labels_df.head()

## 5. Discover Image Files

In [ ]:
SUPPORTED_EXTS = {'.png', '.jpg', '.jpeg', '.bmp', '.tiff', '.tif', '.webp'}

image_paths = sorted([
    p for p in image_dir.iterdir()
    if p.suffix.lower() in SUPPORTED_EXTS
])

print(f"Found {len(image_paths)} images in {image_dir}")

# Warn about images missing from CSV
missing_labels = [p.name for p in image_paths if p.name not in label_map]
if missing_labels:
    print(f"⚠️  {len(missing_labels)} images have no entry in labels.csv (will still be classified)")
    print("   First few:", missing_labels[:5])

## 6. Load MediculaMultiTaskModel

In [ ]:
# ─── Model initialisation ────────────────────────────────────────────────
# MediculaMultiTaskModel uses a Swin encoder with task-specific heads.
# Adapt the constructor arguments to your MeViS version / checkpoint.

model = MediculaMultiTaskModel(
    task=MODEL_TASK,
    num_classes=NUM_CLASSES,
    # pretrained=True,        # uncomment if a pretrained checkpoint is available
    # checkpoint_path=None,   # or pass an explicit path
)

model = model.to(DEVICE)
model.eval()
print("✅ Model loaded and set to eval mode")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Image Preprocessing Pipeline

In [ ]:
preprocess = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    # ImageNet-style normalisation — adjust if your model uses different stats
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    ),
])

def load_image(path: Path) -> torch.Tensor:
    """Load an image file and apply the preprocessing pipeline."""
    img = Image.open(path).convert('RGB')
    return preprocess(img)

print("Preprocessing pipeline ready.")

## 8. Run Inference

In [ ]:
results = []

def chunked(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

print(f"Running inference on {len(image_paths)} images (batch_size={BATCH_SIZE})...")

with torch.inference_mode():
    for batch_paths in chunked(image_paths, BATCH_SIZE):
        tensors, valid_paths = [], []

        for p in batch_paths:
            try:
                tensors.append(load_image(p))
                valid_paths.append(p)
            except Exception as e:
                print(f"  ⚠️  Could not load {p.name}: {e}")

        if not tensors:
            continue

        batch_tensor = torch.stack(tensors).to(model.device)  # (B, 3, H, W)

        # MMM encoder -> feature pyramid -> hidden vector -> classifier
        feature_pyramid = encoder(batch_tensor)
        hidden = nn.Flatten(1)(squeezer(feature_pyramid)[1])  # (B, D)
        logits = clf_head(hidden)                              # (B, num_classes)
        probs  = F.softmax(logits, dim=-1).cpu().numpy()

        for path, prob_vec in zip(valid_paths, probs):
            pred_idx   = int(np.argmax(prob_vec))
            pred_label = SHAPE_CLASSES[pred_idx]
            confidence = float(prob_vec[pred_idx])
            gt_label   = label_map.get(path.name, 'N/A')
            correct    = (pred_label == gt_label) if gt_label != 'N/A' else None

            results.append({
                'image_name'  : path.name,
                'ground_truth': gt_label,
                'prediction'  : pred_label,
                'confidence'  : round(confidence, 4),
                'correct'     : correct,
                **{f'p_{cls}': round(float(p), 4) for cls, p in zip(SHAPE_CLASSES, prob_vec)},
            })

results_df = pd.DataFrame(results)
print(f"\n✅ Inference complete. {len(results_df)} images processed.")
results_df.head(10)


## 9. Accuracy Metrics

In [ ]:
# Only evaluate rows that have a ground-truth label
eval_df = results_df[results_df['ground_truth'] != 'N/A'].copy()

if len(eval_df) == 0:
    print("No images matched labels.csv — skipping accuracy computation.")
else:
    overall_acc = eval_df['correct'].mean()
    print(f"Overall accuracy : {overall_acc:.2%}  ({eval_df['correct'].sum()}/{len(eval_df)})")
    print()

    # Per-class accuracy
    per_class = (
        eval_df.groupby('ground_truth')['correct']
        .agg(['sum', 'count'])
        .rename(columns={'sum': 'correct', 'count': 'total'})
    )
    per_class['accuracy'] = per_class['correct'] / per_class['total']
    print("Per-class accuracy:")
    print(per_class.sort_values('accuracy').to_string())

## 10. Confusion Matrix

In [ ]:
if len(eval_df) > 0:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    present_classes = sorted(eval_df['ground_truth'].unique())
    cm = confusion_matrix(eval_df['ground_truth'], eval_df['prediction'], labels=present_classes)

    fig, ax = plt.subplots(figsize=(max(6, len(present_classes)), max(5, len(present_classes))))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=present_classes)
    disp.plot(ax=ax, colorbar=True, cmap='Blues')
    ax.set_title('Confusion Matrix — Shape Classification', fontsize=14)
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    print("Saved confusion_matrix.png")

## 11. Sample Predictions Visualisation

In [ ]:
N_SHOW = min(16, len(results_df))
sample = results_df.sample(N_SHOW, random_state=42).reset_index(drop=True)

cols = 4
rows = (N_SHOW + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
axes = axes.flatten()

for i, row in sample.iterrows():
    img_path = image_dir / row['image_name']
    try:
        img = mpimg.imread(img_path)
        axes[i].imshow(img, cmap='gray' if img.ndim == 2 else None)
    except Exception:
        axes[i].text(0.5, 0.5, 'Load\nerror', ha='center', va='center')

    gt  = row['ground_truth']
    pred = row['prediction']
    conf = row['confidence']
    colour = 'green' if row['correct'] else ('red' if row['correct'] is False else 'orange')

    axes[i].set_title(
        f"GT: {gt}\nPred: {pred} ({conf:.0%})",
        color=colour, fontsize=9
    )
    axes[i].axis('off')

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Sample Predictions (green=correct, red=wrong)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved sample_predictions.png")

## 12. Confidence Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Overall confidence histogram
axes[0].hist(results_df['confidence'], bins=20, edgecolor='black', color='steelblue')
axes[0].set_xlabel('Confidence')
axes[0].set_ylabel('Count')
axes[0].set_title('Prediction Confidence Distribution')

# Confidence by correctness (if labels available)
if len(eval_df) > 0:
    for label, group in eval_df.groupby('correct'):
        axes[1].hist(group['confidence'], bins=15, alpha=0.6,
                     label='Correct' if label else 'Wrong', edgecolor='black')
    axes[1].legend()
    axes[1].set_xlabel('Confidence')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Confidence: Correct vs Wrong Predictions')

plt.tight_layout()
plt.savefig('confidence_distribution.png', dpi=150)
plt.show()

## 13. Save Results to CSV

In [ ]:
results_df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Results saved to {OUTPUT_CSV}")
print(f"   Columns: {list(results_df.columns)}")
results_df.tail(5)

## 14. Quick Summary

In [ ]:
print("=" * 50)
print("CLASSIFICATION SUMMARY")
print("=" * 50)
print(f"Images processed : {len(results_df)}")
print(f"Classes          : {SHAPE_CLASSES}")

pred_counts = results_df['prediction'].value_counts()
print("\nPrediction distribution:")
for cls, cnt in pred_counts.items():
    bar = '█' * int(cnt / len(results_df) * 30)
    print(f"  {cls:<12} {cnt:>4}  {bar}")

if len(eval_df) > 0:
    print(f"\nOverall accuracy : {eval_df['correct'].mean():.2%}")
    print(f"Mean confidence  : {results_df['confidence'].mean():.2%}")